# Week 23, Day 4: CSV in depth

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w023d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import csv, io

text = 'name;town;note\nAda;Leeds;"likes ; semicolons"\n'
print(list(csv.reader(io.StringIO(text))))
print(list(csv.reader(io.StringIO(text), delimiter=";")))

In [ ]:
import csv, io

text = 'id\tcomment\n1\t"two\nlines"\n2\t"she said ""hi"""\n'
for row in csv.reader(io.StringIO(text), delimiter="\t"):
    print(row)

out = io.StringIO()
w = csv.writer(out, quoting=csv.QUOTE_NONNUMERIC, lineterminator="\n")
w.writerow(["Ada", 36, 1.5, 'say "hi"'])
print(out.getvalue())

In [ ]:
import csv

sample = "name;age;town\nAda;36;Leeds\nBen;25;Bath\n"
dialect = csv.Sniffer().sniff(sample)
print(repr(dialect.delimiter), csv.Sniffer().has_header(sample))

These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('from_excel.csv', 'w', encoding='utf-8') as f:
    f.write('\ufeffname,town\nAdébáyọ̀,Ìbàdàn\nZoë,Leeds\n')

In [ ]:
import csv

with open("from_excel.csv", encoding="utf-8", newline="") as f:
    print(next(csv.DictReader(f)).keys())
with open("from_excel.csv", encoding="utf-8-sig", newline="") as f:
    for row in csv.DictReader(f):
        print(row["name"], row["town"])

These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('marks.csv', 'w', encoding='utf-8') as f:
    f.write('student,mark,date\nAda, 71 ,2026-10-01\nBen,,2026-10-02\nCy,eighty,2026-10-02\nDee,64,02/10/2026\nEve,105,2026-10-03\nFin,58,2026-10-03\n')

In [ ]:
import csv
from datetime import date

def read_marks(path):
    rows, problems = [], []
    with open(path, encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            line = reader.line_num
            row = {k: (v.strip() or None) for k, v in row.items()}
            try:
                mark = int(row["mark"]) if row["mark"] is not None else None
                if mark is not None and not 0 <= mark <= 100:
                    raise ValueError(f"mark {mark} out of range")
                when = date.fromisoformat(row["date"])
            except ValueError as e:
                problems.append(f"line {line}: {e}")
                continue
            rows.append({"student": row["student"], "mark": mark, "date": when})
    return rows, problems

rows, problems = read_marks("marks.csv")
print([(r["student"], r["mark"]) for r in rows])
for p in problems:
    print(p)

> **Common mistake.** Excel "helps" with CSV: it turns `00123` into `123` (dropping leading zeros from IDs and phone numbers), long numbers into `1.23E+15`, and things like `1-2` or `MARCH1` into dates. When CSV has been through Excel, check IDs and codes carefully, and keep such columns as text in your own code (don't convert them to `int`).

In [ ]:
import csv, io, statistics
from collections import defaultdict

text = "class,student,mark\n9A,Ada,71\n9B,Ben,64\n9A,Cy,88\n9B,Dee,70\n9A,Eve,59\n"
by_class = defaultdict(list)
for row in csv.DictReader(io.StringIO(text)):
    by_class[row["class"]].append(int(row["mark"]))

for cls, marks in sorted(by_class.items()):
    print(cls, len(marks), statistics.mean(marks), max(marks))

In [ ]:
import csv, io

students = "id,name\n7,Ada\n12,Ben\n15,Cy\n"
marks = "id,subject,mark\n7,maths,71\n12,maths,64\n7,art,88\n99,art,50\n"

names = {row["id"]: row["name"] for row in csv.DictReader(io.StringIO(students))}
for row in csv.DictReader(io.StringIO(marks)):
    print(names.get(row["id"], "unknown"), row["subject"], row["mark"])

In [ ]:
import csv, io, json

text = "name,age\nAda,36\nBen,25\n"
records = [{"name": r["name"], "age": int(r["age"])} for r in csv.DictReader(io.StringIO(text))]
print(json.dumps(records))

out = io.StringIO()
writer = csv.DictWriter(out, fieldnames=["name", "age"], lineterminator="\n")
writer.writeheader()
writer.writerows(records)
print(out.getvalue())

> **Key idea.** Tell the `csv` module the dialect (`delimiter=`, `quoting=`), or sniff it. Open with `newline=""`, and read files from outside with `encoding="utf-8-sig"` to drop Excel's BOM. Strip, treat blanks as missing, convert types, and collect problems with `reader.line_num`. Summarise with `defaultdict`, join with a dictionary keyed on the shared column, and process big files row by row.

### Exercise: A European file

`EU_TEXT` is a CSV export with `;` as the delimiter and `,` as the decimal point (as in much of Europe). Write `read_prices(text)` returning a list of `(item, price)` pairs with prices as floats, then `to_uk(text)` returning the same table as standard comma-separated text with `.` decimals (header included, `\n` line endings, prices written as in the input but with `.`).

In [ ]:
import csv, io

EU_TEXT = "item;price\nKäse;4,50\nBrot;2,15\n\"Wein; rot\";12,00\n"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert read_prices(EU_TEXT) == [(\"K\u00e4se\", 4.5), (\"Brot\", 2.15), (\"Wein; rot\", 12.0)]\n    assert to_uk(EU_TEXT) == \"item,price\\nK\u00e4se,4.50\\nBrot,2.15\\nWein; rot,12.00\\n\"")

### Exercise: The invisible character

Write `headers(path)` returning the list of column names from a CSV file's first row, opened so that an Excel BOM doesn't end up in the first name, and `has_bom(path)` returning `True` if the file starts with a BOM (read the first character with plain `utf-8` and compare with `"﻿"`).

In [ ]:
# Run this cell first: it creates the files used below.
with open('excel.csv', 'w', encoding='utf-8') as f:
    f.write('\ufeffid,name,town\n1,Ada,Leeds\n')
with open('plain.csv', 'w', encoding='utf-8') as f:
    f.write('id,name,town\n1,Ada,Leeds\n')

In [ ]:
import csv

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert headers(\"excel.csv\") == [\"id\", \"name\", \"town\"] and headers(\"plain.csv\") == [\"id\", \"name\", \"town\"]\n    assert has_bom(\"excel.csv\") is True and has_bom(\"plain.csv\") is False")

### Exercise: Clean and check

Write `load_orders(text)` that reads CSV text with columns `order_id,customer,quantity,price` and returns `(orders, problems)`. Strip spaces from every field. Each good order is a dictionary with `order_id` kept as **text** (it has leading zeros), `customer`, `quantity` as an int of at least 1, and `price` as a float. A row with a missing field, a quantity that isn't a positive whole number, or a price that isn't a number goes into `problems` as `"line N: REASON"` instead, where N is `reader.line_num` and REASON is `"missing customer"` (or whichever field is first missing), `"bad quantity"` or `"bad price"`.

In [ ]:
import csv, io

In [ ]:
# Check: run the exercise cell above first, then this one.
check("TEXT = \"\"\"order_id,customer,quantity,price\n00017, Ada ,2,4.50\n00018,,1,3.00\n00019,Ben,two,1.00\n00020,Cy,0,2.00\n00021,Dee,3,abc\n00022,Eve, 1 ,10\n\"\"\"\ndef test_values():\n    \"several\"\n    orders, problems = load_orders(TEXT)\n    assert orders == [{\"order_id\": \"00017\", \"customer\": \"Ada\", \"quantity\": 2, \"price\": 4.5},\n                      {\"order_id\": \"00022\", \"customer\": \"Eve\", \"quantity\": 1, \"price\": 10.0}]\n    assert problems == [\"line 3: missing customer\", \"line 4: bad quantity\", \"line 5: bad quantity\", \"line 6: bad price\"]")

### Exercise: A pivot table

Write `pivot(text)` for CSV with columns `region,month,sales`, returning a dictionary from each region to a dictionary from each month to the total sales (as an int), plus `"total"` for the region's total. Then `pivot_csv(text)` returning that as CSV text: a header `region,` then the months in the order they first appear, then `total`; one row per region in alphabetical order (0 for months with no sales); `\n` line endings.

In [ ]:
import csv, io
from collections import defaultdict

In [ ]:
# Check: run the exercise cell above first, then this one.
check("TEXT = \"region,month,sales\\nNorth,Jan,100\\nSouth,Jan,80\\nNorth,Feb,120\\nNorth,Jan,30\\nSouth,Mar,50\\n\"\ndef test_values():\n    \"several\"\n    assert pivot(TEXT) == {\"North\": {\"Jan\": 130, \"Feb\": 120, \"total\": 250}, \"South\": {\"Jan\": 80, \"Mar\": 50, \"total\": 130}}\n    assert pivot_csv(TEXT) == \"region,Jan,Feb,Mar,total\\nNorth,130,120,0,250\\nSouth,80,0,50,130\\n\"")

### Exercise: Join two tables

Write `join(left_text, right_text, key)` that joins two CSV texts on the column `key` and returns a list of dictionaries combining each right row with the matching left row's other columns (the right row's values win if both have a column of the same name). Right rows with no match get `None` for the left's columns. Build a dictionary from the left table first, so it's O(n + m).

In [ ]:
import csv, io

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    students = \"id,name,class\\n7,Ada,9A\\n12,Ben,9B\\n\"\n    marks = \"id,subject,mark\\n7,maths,71\\n12,maths,64\\n99,art,50\\n\"\n    assert join(students, marks, \"id\") == [\n        {\"id\": \"7\", \"subject\": \"maths\", \"mark\": \"71\", \"name\": \"Ada\", \"class\": \"9A\"},\n        {\"id\": \"12\", \"subject\": \"maths\", \"mark\": \"64\", \"name\": \"Ben\", \"class\": \"9B\"},\n        {\"id\": \"99\", \"subject\": \"art\", \"mark\": \"50\", \"name\": None, \"class\": None},\n    ]")

### Bug hunt: Split on commas

This reads a CSV line by splitting on commas, so the address `"4 High St, Leeds"` becomes two fields and the town ends up as the third. Use the `csv` module instead, so it prints `['Ada', '4 High St, Leeds', 'LS1 4AB']`.

In [ ]:
line = 'Ada,"4 High St, Leeds",LS1 4AB'
print(line.split(","))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"fixed\"\n    expect_output(\"['Ada', '4 High St, Leeds', 'LS1 4AB']\")\n    assert \"csv\" in source")

### Bug hunt: Lost leading zeros

This tidies a list of staff IDs, but converting them to `int` loses the leading zeros, and `"00417"` becomes `417`. IDs are labels, not quantities. Fix it so the IDs stay as text and it prints `['00417', '01203', '10000']` (sorted).

In [ ]:
import csv, io

text = "id,name\n01203,Ben\n00417,Ada\n10000,Cy\n"
ids = sorted(int(row["id"]) for row in csv.DictReader(io.StringIO(text)))
print(ids)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"fixed\"\n    expect_output(\"['00417', '01203', '10000']\")")

### Puzzle: Read a file from Excel

Arrange the lines so the program reads a semicolon-separated file saved by Excel and prints each name. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them

Work these out without running them.

In [ ]:
import csv

a = ___   # next(csv.reader(['a,"b,c",d']))
b = ___   # next(csv.reader(['x|y|z'], delimiter="|"))
c = ___   # next(csv.reader(['"say ""hi"""']))
d = ___   # the encoding that removes Excel's byte order mark (a string)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert a == [\"a\", \"b,c\", \"d\"] and b == [\"x\", \"y\", \"z\"] and c == ['say \"hi\"'] and d == \"utf-8-sig\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: merge monthly exports

A shop exports a CSV of sales each month, but the exports aren't consistent: some use `,` and some `;`; some were saved by Excel with a BOM; column names vary in capitals and spaces (`Date`, ` date`, `DATE`); and amounts may use `,` as the decimal point. Write `merge_exports(paths)` that reads every file (sniffing the delimiter from the first line), normalises column names (strip and lower-case), keeps the columns `date`, `item` and `amount`, converts amounts to floats (accepting `,` or `.` as the decimal point), drops rows where any of the three is empty, removes exact duplicate rows (the same date, item and amount, which happens when exports overlap), and returns the rows as `(date, item, amount)` tuples sorted by date, then item. Then `write_merged(paths, out_path)` that writes them to one standard CSV file (UTF-8, comma-separated, header `date,item,amount`, amounts with 2 decimal places, `\n` line endings) and returns the number of rows written.

In [ ]:
# Run this cell first: it creates the files used below.
with open('oct.csv', 'w', encoding='utf-8') as f:
    f.write('\ufeffDate;Item;Amount\n2026-10-02;Tea;4,50\n2026-10-01;Jam;3,10\n2026-10-03;Bread;\n')
with open('nov.csv', 'w', encoding='utf-8') as f:
    f.write(' date , ITEM , amount\n2026-11-01,Tea,4.50\n2026-10-02,Tea,4.5\n2026-11-02,Cake,6\n')

In [ ]:
import csv

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_merge():\n    \"merge\"\n    rows = merge_exports([\"oct.csv\", \"nov.csv\"])\n    assert rows == [(\"2026-10-01\", \"Jam\", 3.1), (\"2026-10-02\", \"Tea\", 4.5), (\"2026-11-01\", \"Tea\", 4.5), (\"2026-11-02\", \"Cake\", 6.0)]\ndef test_write():\n    \"write\"\n    assert write_merged([\"oct.csv\", \"nov.csv\"], \"all.csv\") == 4\n    assert read_file(\"all.csv\") == \"date,item,amount\\n2026-10-01,Jam,3.10\\n2026-10-02,Tea,4.50\\n2026-11-01,Tea,4.50\\n2026-11-02,Cake,6.00\\n\"")

## Warm-up

The same file, read two ways. With the default comma, each line is one field; told the delimiter is `;`, it comes out properly, and the `;` inside the quoted note stays where it belongs. (`io.StringIO` makes a string behave like an open file, handy for examples and tests.)

## Dialects

"CSV" really means a family of formats. The `csv` module calls each variation a **dialect**, described by a few settings:

| Setting | Means | Common values |
|---|---|---|
| `delimiter` | what separates fields | `,` (default), `;` (much of Europe, where `,` is the decimal point), `\t` (tab-separated, "TSV"), `|` |
| `quotechar` | what surrounds fields containing special characters | `"` (default) |
| `quoting` | when to quote when **writing** | `csv.QUOTE_MINIMAL` (default: only when needed), `QUOTE_ALL`, `QUOTE_NONNUMERIC`, `QUOTE_NONE` |
| `lineterminator` | line ending when writing | `"\r\n"` (default) |

Pass the settings to `csv.reader`, `csv.writer`, `DictReader` and `DictWriter`. Inside quotes, a field can contain the delimiter, a line break, or a quote mark written twice (`""`):

That's why you never read CSV with `line.split(",")` or loop over lines yourself: a single record can span several lines.

**Not sure of the format?** `csv.Sniffer().sniff(sample)` guesses the dialect from a sample of the text, and `has_header(sample)` guesses whether the first row is a header:

Sniffing is a guess: use it when files come from many places, and say the dialect explicitly when you know it.

## Encodings and the BOM

Files are bytes; an **encoding** says how bytes become characters (Week 11). For CSV:

- Open files with `encoding="utf-8"` and `newline=""` (the `csv` module handles line endings itself, and `newline=""` stops Python interfering, which matters for line breaks inside fields).
- Excel often saves "CSV UTF-8" with a hidden **byte order mark** (BOM) at the very start. Read with plain `utf-8`, it sticks to the first header: you get `"﻿name"` instead of `"name"`, and `row["name"]` fails mysteriously. Open such files with **`encoding="utf-8-sig"`**, which removes the BOM if it's there and does nothing if it isn't. Use it for every CSV you read from outside.
- Writing for Excel users, `utf-8-sig` also makes Excel show accented letters correctly.
- Older files may be in `cp1252` or `latin-1`; garbled letters like "Ã©" instead of "é" are the sign of a wrong guess.

## Clean, convert and check

Every field in a CSV is **text**. Real data also has gaps, stray spaces, and values in the wrong form. A careful reader:

1. strips spaces from every field;
2. treats empty fields (and things like `"N/A"`) as missing (`None`);
3. converts each field to the right type, and
4. **collects** problems with their line numbers, instead of crashing at the first bad row, so you can fix them all at once.

`reader.line_num` is the line number in the file (counting the header as line 1), which is exactly what someone fixing the file needs.

## Summarise and join

Once rows are dictionaries, everything from Weeks 10 and 20 applies. **Group and summarise** with `defaultdict`:

**Join** two tables on a shared column, like a spreadsheet's VLOOKUP or a database join: build a dictionary from one table keyed by that column, then look each row of the other up in it. That's O(n + m), where comparing every pair would be O(n × m):

## Big files, and CSV to JSON

`csv.reader` and `DictReader` read **one row at a time**, so a loop over them handles files far bigger than memory, as long as you don't collect every row into a list. Keep running totals instead:

```
total = count = 0
with open("huge.csv", encoding="utf-8-sig", newline="") as f:
    for row in csv.DictReader(f):
        total += float(row["amount"])
        count += 1
```

Converting between CSV and JSON is a common job: CSV for spreadsheets, JSON for web services.

For big numerical tables, the third-party `pandas` library reads CSV into a table in one line and summarises it in another; it's built on exactly these ideas, and you'll meet it in Phase 5.

## Practice

## Quiz

## Daily challenge

## Before you go

Export a small table from any spreadsheet program you have as CSV, open it in a text editor to see exactly what was saved (delimiter, quotes, BOM), and read it with today's careful reader. Tomorrow: dates, times and time zones, the last piece of real-world data that catches everyone out.